<a href="https://colab.research.google.com/github/Kirill-LN/Practicum/blob/main/%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5_%D0%B7%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D1%8F_%D0%B4%D0%BB%D1%8F_%D0%BF%D0%B5%D1%80%D0%B2%D0%BE%D0%B3%D0%BE_%D0%B1%D0%BB%D0%BE%D0%BA%D0%B0_%D0%BF%D0%BE_%D0%B1%D0%B0%D0%B7%D0%BE%D0%B2%D0%BE%D0%BC%D1%83_%D0%BA%D1%83%D1%80%D1%81%D1%83_%D0%BF%D0%BE_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практические задания для первого блока по базовому курсу по Python

## Задание 1: Анализ данных из текстового файла

In [4]:
import logging
import pandas as pd
from openpyxl import load_workbook

logging.basicConfig(
    filename='app.log',
    filemode='w',
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True,
    encoding='UTF-8',
)


def count_product(path_txt_file):
    """ Функция обработки данных из файла.

    Читает данные их файла data.txt, преобразует данные из списка в словарь, проводит расчеты дохода по товарам и общего дохода.
    Возвращает словарь и общий доход.
    """

    with open(path_txt_file, 'r', encoding='UTF-8') as file:
        content = file.readlines()
        logging.info('Чтение файла прошло успешно')
        products_data = {}

        for row in content:
            product, count, price = row.strip().split(',')
            products_data[product] ={
                'count': int(count),
                'price': float(price),
            }
        logging.info(f'Обработано товаров: {len(products_data)}')
        total_income = 0

        for data in products_data.values():
            total_revenue_product = data['count'] * data['price']
            data['total_revenue_product'] = round(total_revenue_product, 2)
            total_income += data['total_revenue_product']
            total_income = round(total_income, 2)

        logging.info(f'Расчет дохода завершен. Он составляет : {total_income}')

        return products_data, total_income


def create_excel_table(products_data, total_income):
    """ Функция создания excel таблицы с результатами расчета по данным файла data.txt .

    Принимает словарь с данными и общий доход, формирует DataFrame из списка, записывает DataFrame в excel файл.
    Возвращает excel таблицу с комментарием после последний сроки о результатах общих продаж.
    """

    logging.info('Создание Excel таблицы')

    # Создаем DataFrame
    df = pd.DataFrame.from_dict(
        products_data,
        orient='index'
    )

    df.index.name = 'Товар'

    df = df.rename(columns={
    'count': 'Количество',
    'price': 'Цена',
    'total_revenue_product': 'Выручка'
    }
    )

    #  Создаем и заполняем Excel файл
    with pd.ExcelWriter('table_output.xlsx') as writer:
        df.to_excel(writer, index=True)

    wb = load_workbook('table_output.xlsx')
    sheet = wb.active


    if total_income > 1000000:
        message = f'Общий доход равен : {total_income}, хорошо заработали))'
    else:
        message = f'Общий доход равен : {total_income}, в следующий раз надо постараться'

    message_row = sheet.max_row + 1

    sheet.cell(
        row=message_row,  # Строка для сообщения
        column=1,  # Колонка
        value=message,  # Сообщение
    )

    wb.save('table_output.xlsx')
    logging.info('Excel файл сохранен успешно')

    # Читаем excel файл и возвращаем результат
    with pd.ExcelFile('table_output.xlsx') as xls:
        df = pd.read_excel(xls, 'Sheet1')

    return (df)


path_txt_file = 'data.txt'

try:
    products_data, total_income = count_product(path_txt_file)
    print(create_excel_table(products_data, total_income))

except FileNotFoundError as e:
    logging.critical(f'Ошибка, файл не найден: {e}')
except TypeError as e:
    logging.critical(f'Ошибка обработки данных: {e}')
except ValueError as e:
    logging.error(f'Ошибка формата данных: {e}')



                                               Товар  Количество      Цена  \
0                                          Телевизор         5.0  29999.99   
1                                        Холодильник         2.0  49999.50   
2                                           Смартфон        10.0  19999.00   
3                                            Ноутбук         3.0  69999.99   
4                                            Пылесос         7.0   8999.90   
5                                  Стиральная машина         4.0  35999.95   
6                                      Микроволновка         8.0   7999.99   
7                                          Кофеварка        15.0   4999.50   
8  Общий доход равен : 1005980.44, хорошо заработ...         NaN       NaN   

     Выручка  
0  149999.95  
1   99999.00  
2  199990.00  
3  209999.97  
4   62999.30  
5  143999.80  
6   63999.92  
7   74992.50  
8        NaN  
